In [ ]:
# Install dependencies and download the Cornell Movie Dialogs Corpus dataset
!pip install datasets torch transformers
!wget http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
!unzip -qq cornell_movie_dialogs_corpus.zip
!rm cornell_movie_dialogs_corpus.zip
!mkdir datasets
!mv cornell\ movie-dialogs\ corpus/movie_conversations.txt ./datasets
!mv cornell\ movie-dialogs\ corpus/movie_lines.txt ./datasets

In [ ]:
# Import standard libraries and PyTorch modules for building the Transformer
from collections import Counter
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.utils.data
import math
import torch.nn.functional as F

# 1) Data Processing

This tutorial trains a <a href="https://arxiv.org/abs/1706.03762" class="external">Transformer model</a> to be a chatbot. This is an advanced example that assumes knowledge of [text generation](https://tensorflow.org/alpha/tutorials/text/text_generation), [attention](https://www.tensorflow.org/alpha/tutorials/text/nmt_with_attention) and [transformer](https://www.tensorflow.org/alpha/tutorials/text/transformer).


We will use the conversations in movies and TV shows provided by [Cornell Movie-Dialogs Corpus](https://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html), which contains more than 220 thousands conversational exchanges between more than 10k pairs of movie characters, as our dataset.

`movie_conversations.txt` contains list of the conversation IDs and `movie_lines.text` contains the text of assoicated with each conversation ID. For further  information regarding the dataset, please check the README file in the zip file.


In [ ]:
# Initialize global parameters for data processing
max_len = 25

# Define a helper function to remove punctuation and lowercase strings
def remove_punc(string):
    punctuations = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    no_punct = ""
    for char in string:
        if char not in punctuations:
            # space is also a character
            no_punct = no_punct + char
    return no_punct.lower()

# Define file paths and load raw dataset files
corpus_movie_conv = './datasets/movie_conversations.txt'
corpus_movie_lines = './datasets/movie_lines.txt'
with open(corpus_movie_conv, 'r', encoding='iso-8859-1') as c:
    conv = c.readlines()
with open(corpus_movie_lines, 'r', encoding='iso-8859-1') as l:
    lines = l.readlines()

# Map line IDs to their corresponding text content
# Output: Dictionary mapping line_id -> text_string
lines_dic = {}
for line in lines:
    objects = line.split(" +++$+++ ")
    lines_dic[objects[0]] = objects[-1]

# Parse conversation sequences and generate question-answer pairs
# Output: List of [question_tokens, answer_tokens], each truncated to max_len
pairs = []
for con in conv:
    ids = eval(con.split(" +++$+++ ")[-1])
    for i in range(len(ids)):
        qa_pairs = []

        if i == len(ids) - 1:
            break

        # Clean and tokenize sentence strings
        first = remove_punc(lines_dic[ids[i]].strip())
        second = remove_punc(lines_dic[ids[i+1]].strip())

        # Truncate sequences to the defined max_len
        # Output: (num_tokens,) limited by max_len
        qa_pairs.append(first.split()[:max_len])
        qa_pairs.append(second.split()[:max_len])
        pairs.append(qa_pairs)

# Print a sample pair to verify processing
print(pairs[20])

[['i', 'really', 'really', 'really', 'wanna', 'go', 'but', 'i', 'cant', 'not', 'unless', 'my', 'sister', 'goes'], ['im', 'workin', 'on', 'it', 'but', 'she', 'doesnt', 'seem', 'to', 'be', 'goin', 'for', 'him']]


In [ ]:
# Initialize vocabulary settings and frequency counter
min_word_freq = 5
word_freq = Counter()

# Count frequency of each word across all conversation pairs
for pair in pairs:
    word_freq.update(pair[0])
    word_freq.update(pair[1])

# Filter words by minimum frequency and build word_map with special tokens
# vocab_size = number of words above threshold + 4 special tokens
words = [w for w in word_freq.keys() if word_freq[w] > min_word_freq]
word_map = {k: v + 1 for v, k in enumerate(words)}
word_map['<unk>'] = len(word_map) + 1
word_map['<start>'] = len(word_map) + 1
word_map['<end>'] = len(word_map) + 1
word_map['<pad>'] = 0

print("Total words are: {}".format(len(word_map)))

# Encode a question (no start/end tokens) into a padded integer sequence
# Input: (num_tokens,) → Output: (max_len,)
def encode_question(words, word_map):
    enc_c = [word_map.get(word, word_map['<unk>']) for word in words] + [word_map['<pad>']] * (max_len - len(words))
    return enc_c

# Encode a reply (with <start> and <end> tokens) into a padded integer sequence
# Input: (num_tokens,) → Output: (max_len + 2,)
def encode_reply(words, word_map):
    enc_c = (
        [word_map['<start>']] + [word_map.get(word, word_map['<unk>']) for word in words] +         [word_map['<end>']] + [word_map['<pad>']] * (max_len - len(words))
    )
    return enc_c

# Encode all conversation pairs into numerical format
# Input: List of string token lists → Output: List of [[max_len], [max_len + 2]]
pairs_encoded = []
for pair in pairs:
    qus = encode_question(pair[0], word_map)
    ans = encode_reply(pair[1], word_map)
    pairs_encoded.append([qus, ans])

Total words are: 18243


In [ ]:
# Define the custom Dataset class for movie conversation pairs
class Dataset(torch.utils.data.Dataset):

    def __init__(self, pairs):
        self.pairs = pairs
        self.dataset_size = len(self.pairs)

    # Retrieve a single encoded question/reply pair as LongTensors
    # Output: question (max_len,), reply (max_len + 2,)
    def __getitem__(self, i):
        question = torch.LongTensor(self.pairs[i][0])
        reply = torch.LongTensor(self.pairs[i][1])
        return question, reply

    def __len__(self):
        return self.dataset_size

# Initialize the DataLoader to manage batches of encoded pairs
# Output per batch: question (batch_num, max_len), reply (batch_num, max_len + 2)
train_loader = DataLoader(Dataset(pairs_encoded), batch_size=32, shuffle=True, pin_memory=True)

# Extract a sample batch to verify tensor shapes
question, reply = next(iter(train_loader))
print("Question: ", question.size())
print("Answer: ", reply.size())

Question:  torch.Size([32, 25])
Answer:  torch.Size([32, 27])


# 2) Masking

1. Mask all the pad tokens (value `0`) in the batch to ensure the model does not treat padding as input.

2. **Look-ahead Mask** to mask the future tokens in a sequence.
We also mask out pad tokens. i.e. To predict the third word, only the first and second word will be used

In [ ]:
# Function to generate padding and look-ahead masks for the encoder and decoder
def create_masks(question, reply_input, reply_target, device='cuda'):

    # Helper: generate a lower-triangular mask to prevent attending to future tokens
    # Input: seq_len (scalar) → Output: (1, seq_len, seq_len)
    def subsequent_mask(size):
        mask = torch.triu(torch.ones(size, size)).transpose(0, 1).type(dtype=torch.uint8)
        return mask.unsqueeze(0)

    # Create a padding mask for the encoder input (True where token != pad)
    # Input: (batch_num, max_len) → Output: (batch_num, 1, 1, max_len)
    question_mask = (question != 0).to(device)
    question_mask = question_mask.unsqueeze(1).unsqueeze(1)

    # Create a padding mask for the decoder input
    # Input: (batch_num, output_len) → Output: (batch_num, 1, output_len)
    reply_input_mask = (reply_input != 0).unsqueeze(1)

    # Combine padding mask with look-ahead mask for decoder self-attention
    # Input: (batch_num, 1, output_len) & (1, output_len, output_len)
    # Output after &: (batch_num, output_len, output_len)
    reply_input_mask = (
        reply_input_mask &
        subsequent_mask(reply_input.size(-1)).type_as(reply_input_mask.data)
    )
    # Add head dimension → (batch_num, 1, output_len, output_len)
    reply_input_mask = reply_input_mask.unsqueeze(1)

    # Create a mask for the decoder target sequence (True where token != pad)
    # Input: (batch_num, output_len) → Output: (batch_num, output_len)
    reply_target_mask = reply_target != 0

    return question_mask, reply_input_mask, reply_target_mask

# Right-shift reply to create teacher-forced decoder input and target
# Input: (batch_num, output_len + 2) → reply_input: (batch_num, output_len + 1), reply_target: (batch_num, output_len + 1)
reply_input = reply[:, :-1]
reply_target = reply[:, 1:]
print('Reply Target Size: ', reply_target.size())

# Generate all required masks for training
# question_mask:     (batch_num, 1, 1, max_len)
# reply_input_mask:  (batch_num, 1, output_len + 1, output_len + 1)
# reply_target_mask: (batch_num, output_len + 1)
question_mask, reply_input_mask, reply_target_mask = create_masks(question, reply_input, reply_target)
print('question_mask Size: ', question_mask.size())
print('reply_input_mask Size: ', reply_input_mask.size())
print('reply_target_mask Size: ', reply_target_mask.size())

Reply Target Size:  torch.Size([32, 26])
question_mask Size:  torch.Size([32, 1, 1, 25])
reply_input_mask Size:  torch.Size([32, 1, 26, 26])
reply_target_mask Size:  torch.Size([32, 26])


# 3) Embedding

## 3.1 Positional Embedding
Since this model doesn't contain any recurrence or convolution, positional encoding is added to give the model some information about the relative position of the words in the sentence.

The positional encoding vector is added to the embedding vector. Embeddings represent a token in a d-dimensional space where tokens with similar meaning will be closer to each other. But the embeddings do not encode the relative position of words in a sentence. So after adding the positional encoding, words will be closer to each other based on the *similarity of their meaning and their position in the sentence*, in the d-dimensional space.

See the notebook on [positional encoding](https://github.com/tensorflow/examples/blob/master/community/en/position_encoding.ipynb) to learn more about it. The formula for calculating the positional encoding is as follows:

$$\Large{PE_{(pos, 2i)} = sin(pos / 10000^{2i / d_{model}})} $$
$$\Large{PE_{(pos, 2i+1)} = cos(pos / 10000^{2i / d_{model}})} $$

In [ ]:
# Positional Embedding: combines token embeddings, sinusoidal positional encoding, and layer encoding
class Embeddings(nn.Module):
    """
    Implements embeddings of the words and adds their positional encodings.
    """
    def __init__(self, vocab_size, d_model, max_len=50, num_layers=6):
        super(Embeddings, self).__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(0.1)
        # Token embedding lookup table
        # Input: (batch_num, seq_len) → Output: (batch_num, seq_len, model_dim)
        self.embed = nn.Embedding(vocab_size, d_model)

        # Sinusoidal positional encoding for sequence positions
        # Output: (1, max_len, model_dim)
        self.pe = self.create_positional_encoding(max_len, self.d_model)

        # Layer-level positional encoding (Temporal Embedding)
        # Output: (1, num_layers, model_dim)
        self.te = self.create_positional_encoding(num_layers, self.d_model)
        self.dropout = nn.Dropout(0.1)

    def create_positional_encoding(self, max_len, d_model, device='cpu'):
        # Initialize the encoding matrix with zeros
        # Output: (max_len, model_dim)
        pe = torch.zeros(max_len, d_model).to(device)

        # Fill with sine/cosine values based on position and dimension index
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1))/d_model)))

        # Unsqueeze to add a batch dimension
        # Input: (max_len, model_dim) → Output: (1, max_len, model_dim)
        pe = pe.unsqueeze(0)
        return pe

    def forward(self, embedding, layer_idx):
        # On the first layer, apply token embedding and scale by sqrt(model_dim)
        # Input: (batch_num, seq_len) → Output: (batch_num, seq_len, model_dim)
        if layer_idx == 0:
            embedding = self.embed(embedding) * math.sqrt(self.d_model)

        # Add sinusoidal positional encoding
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, model_dim)
        embedding += self.pe[:, :embedding.size(1)]

        # Add layer-level (temporal) encoding broadcast across all positions
        # te slice: (1, model_dim) → unsqueeze+repeat → (1, seq_len, model_dim)
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, model_dim)
        embedding += (
            self.te[:, layer_idx, :]
            .unsqueeze(1)
            .repeat(1, embedding.size(1), 1)
        )

        # Apply dropout regularization
        # Input/Output: (batch_num, seq_len, model_dim)
        embedding = self.dropout(embedding)
        return embedding

# Instantiate the embedding model
embed_model = Embeddings(len(word_map), 512)

# Test the embedding forward pass
# Input: (batch_num, max_len) → Output: (batch_num, max_len, model_dim)
result = embed_model.forward(question, 0)
print(result.size())

torch.Size([32, 25, 512])


# 4) Transformers

## 4.1 Scaled dot product Attention

The scaled dot-product attention function used by the transformer takes three inputs: Q (query), K (key), V (value). The equation used to calculate the attention weights is:

$$\Large{Attention(Q, K, V) = softmax_k(\frac{QK^T}{\sqrt{d_k}} + M) V} $$

As the softmax normalization is done on the `key`, its values decide the amount of importance given to the `query`.

The output represents the multiplication of the attention weights and the `value` vector. This ensures that the words we want to focus on are kept as is and the irrelevant words are flushed out.

The dot-product attention is scaled by a factor of square root of the depth. This is done because for large values of depth, the dot product grows large in magnitude pushing the softmax function where it has small gradients resulting in a very hard softmax.

For example, consider that `query` and `key` have a mean of 0 and variance of 1. Their matrix multiplication will have a mean of 0 and variance of `dk`. Hence, *square root of `dk`* is used for scaling (and not any other number) because the matmul of `query` and `key` should have a mean of 0 and variance of 1, so that we get a gentler softmax.

Masking is needed to prevent the attention mechanism of a transformer from “cheating” in the decoder or peeking to the future. The mask is multiplied with *-1e9 (close to negative infinity).* This is done because the mask is summed with the scaled matrix multiplication of `query` and `key` and is applied immediately before a softmax. The goal is to zero out these cells, and large negative inputs to softmax are near zero in the output.

## 4.2 Multi-head attention

<img src="https://www.tensorflow.org/images/tutorials/transformer/multi_head_attention.png" width="500" alt="multi-head attention">

Multi-head attention consists of four parts:
* Linear layers and split into heads.
* Scaled dot-product attention.
* Concatenation of heads.
* Final linear layer.

Each multi-head attention block gets three inputs; Q (query), K (key), V (value). These are put through linear (Dense) layers and split up into multiple heads.

The `scaled_dot_product_attention` defined above is applied to each head (broadcasted for efficiency). An appropriate mask must be used in the attention step.  The attention output for each head is then concatenated (using `tf.transpose`, and `tf.reshape`) and put through a final `Dense` layer.

Instead of one single attention head, `query`, `key`, and `value` are split into multiple heads because it allows the model to jointly attend to information at different positions from different representational spaces. After the split each head has a reduced dimensionality, so the total computation cost is the same as a single head attention with full dimensionality.

In [ ]:
# Multi-Head Attention: projects Q/K/V, splits into heads, computes scaled dot-product attention, then recombines
class MultiHeadAttention(nn.Module):

    def __init__(self, heads, d_model, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads   # head dimension = model_dim / num_heads
        self.heads = heads
        self.dropout = nn.Dropout(dropout)
        # Linear projections for Q, K, V and output
        # Each: Input (batch_num, seq_len, model_dim) → Output (batch_num, seq_len, model_dim)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.output_linear = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask):
        # Project inputs to query, key, and value representations
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, model_dim)
        query = self.query(query)
        key = self.key(key)
        value = self.value(value)

        # Reshape and transpose to split into num_heads
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, num_heads, seq_len, d_k)
        query = query.view(query.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        key = key.view(key.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        value = value.view(value.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)

        # Scaled dot-product attention scores
        # Input: (batch_num, num_heads, query_len, d_k) x (batch_num, num_heads, d_k, key_len)
        # Output: (batch_num, num_heads, query_len, key_len)
        scores = torch.matmul(query, key.permute(0, 1, 3, 2)) / math.sqrt(query.size(-1))

        # Mask out padding/future positions with -1e9 before softmax
        # Input/Output: (batch_num, num_heads, query_len, key_len)
        scores = scores.masked_fill(mask == 0, -1e9)

        # Convert scores to attention weights via softmax
        # Output: (batch_num, num_heads, query_len, key_len)
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Weighted sum over value vectors
        # Input: (batch_num, num_heads, query_len, key_len) x (batch_num, num_heads, key_len, d_k)
        # Output: (batch_num, num_heads, query_len, d_k)
        context = torch.matmul(weights, value)

        # Merge heads back: transpose then reshape
        # Input: (batch_num, num_heads, query_len, d_k) → Output: (batch_num, query_len, model_dim)
        context = (
            context.permute(0, 2, 1, 3)
            .contiguous()
            .view(context.shape[0], -1, self.heads * self.d_k)
        )

        # Final linear projection to model_dim
        # Input: (batch_num, query_len, model_dim) → Output: (batch_num, query_len, model_dim)
        return self.output_linear(context)

## 4.3 FeedForward Layer

Network with one hidden layer that applies a non-linear activation function (GELU) to the output of the first linear layer

In [ ]:
# Position-wise Feed-Forward Network: two linear layers with GELU activation and dropout
class FeedForward(nn.Module):
    def __init__(self, d_model, middle_dim=2048):
        super(FeedForward, self).__init__()
        # First linear expansion: model_dim → middle_dim
        self.fc1 = nn.Linear(d_model, middle_dim)
        # Second linear projection: middle_dim → model_dim
        self.fc2 = nn.Linear(middle_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        self.activation = torch.nn.GELU()

    def forward(self, x):
        # First linear layer with GELU activation
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, middle_dim)
        out = self.activation(self.fc1(x))

        # Dropout then second linear layer to project back to model_dim
        # Input: (batch_num, seq_len, middle_dim) → Output: (batch_num, seq_len, model_dim)
        out = self.fc2(self.dropout(out))

        return out

## 4.4 Encoder Layer

Each encoder layer consists of sublayers:

1. Multi-head attention (with padding mask)
2. 2 dense layers followed by dropout

Each of these sublayers has a residual connection around it followed by a layer normalization. Residual connections help in avoiding the vanishing gradient problem in deep networks.

The output of each sublayer is `LayerNorm(x + Sublayer(x))`. The normalization is done on the `d_model` (last) axis.

In [ ]:
# Encoder Layer: self-attention + feed-forward with residual connections and layer normalization
class EncoderLayer(nn.Module):

    def __init__(self, d_model, heads):
        super(EncoderLayer, self).__init__()
        self.layernorm = nn.LayerNorm(d_model)
        self.self_multihead = MultiHeadAttention(heads, d_model)
        self.feed_forward = FeedForward(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, mask):
        # Self-attention over encoder inputs
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, model_dim)
        interacted = self.dropout(
            self.self_multihead(embeddings, embeddings, embeddings, mask)
        )

        # First residual connection + layer normalization
        # Input/Output: (batch_num, seq_len, model_dim)
        interacted = self.layernorm(interacted + embeddings)

        # Position-wise feed-forward network
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, model_dim)
        feed_forward_out = self.dropout(
            self.feed_forward(interacted)
        )

        # Second residual connection + layer normalization
        # Input/Output: (batch_num, seq_len, model_dim)
        encoded = self.layernorm(feed_forward_out + interacted)

        return encoded

## 4.5 Decoder Layer

Each decoder layer consists of sublayers:

1.   Masked multi-head attention (with look ahead mask and padding mask)
2.   Multi-head attention (with padding mask). `value` and `key` receive the *encoder output* as inputs. `query` receives the *output from the masked multi-head attention sublayer.*
3.   2 dense layers followed by dropout

Each of these sublayers has a residual connection around it followed by a layer normalization. The output of each sublayer is `LayerNorm(x + Sublayer(x))`. The normalization is done on the `d_model` (last) axis.

As `query` receives the output from decoder's first attention block, and `key` receives the encoder output, the attention weights represent the importance given to the decoder's input based on the encoder's output. In other words, the decoder predicts the next word by looking at the encoder output and self-attending to its own output. See the demonstration above in the scaled dot product attention section.

In [ ]:
# Decoder Layer: masked self-attention + encoder-decoder cross-attention + feed-forward with residuals
class DecoderLayer(nn.Module):

    def __init__(self, d_model, heads):
        super(DecoderLayer, self).__init__()
        self.layernorm = nn.LayerNorm(d_model)
        self.self_multihead = MultiHeadAttention(heads, d_model)    # masked self-attention
        self.src_multihead = MultiHeadAttention(heads, d_model)     # cross-attention
        self.feed_forward = FeedForward(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, encoded, src_mask, target_mask):
        # Masked self-attention over decoder inputs (prevents attending to future tokens)
        # Input: (batch_num, output_len, model_dim) → Output: (batch_num, output_len, model_dim)
        query = self.dropout(
            self.self_multihead(embeddings, embeddings, embeddings, target_mask)
        )

        # First residual connection + layer normalization
        # Input/Output: (batch_num, output_len, model_dim)
        query = self.layernorm(query + embeddings)

        # Cross-attention: decoder queries attend to encoder keys/values
        # query: (batch_num, output_len, model_dim), encoded: (batch_num, input_len, model_dim)
        # Output: (batch_num, output_len, model_dim)
        interacted = self.dropout(
            self.src_multihead(query, encoded, encoded, src_mask)
        )

        # Second residual connection + layer normalization
        # Input/Output: (batch_num, output_len, model_dim)
        interacted = self.layernorm(interacted + query)

        # Position-wise feed-forward network
        # Input: (batch_num, output_len, model_dim) → Output: (batch_num, output_len, model_dim)
        feed_forward_out = self.dropout(
            self.feed_forward(interacted)
        )

        # Third residual connection + layer normalization
        # Input/Output: (batch_num, output_len, model_dim)
        decoded = self.layernorm(feed_forward_out + interacted)

        return decoded

## 4.6 Transformer

Transformer consists of the encoder, decoder and a final linear layer. The output of the decoder is the input to the linear layer and its output is returned.

<img src='https://lilianweng.github.io/posts/2020-04-07-the-transformer-family/transformer.png'>

<img src='https://1.bp.blogspot.com/-AVGK0ApREtk/WaiAuzddKVI/AAAAAAAAB_A/WPV5ropBU-cxrcMpqJBFHg73K9NX4vywwCLcBGAs/s1600/image2.png'>

In [ ]:
# Transformer: stacks shared Embeddings, EncoderLayer, and DecoderLayer with a final vocab projection
class Transformer(nn.Module):

    def __init__(self, d_model, heads, num_layers, word_map):
        super(Transformer, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers
        self.vocab_size = len(word_map)
        # Shared embedding module used by both encoder and decoder
        self.embed = Embeddings(self.vocab_size, d_model, num_layers=num_layers)
        self.encoder = EncoderLayer(d_model, heads)
        self.decoder = DecoderLayer(d_model, heads)
        # Final linear projection to vocabulary logits
        # Input: (batch_num, seq_len, model_dim) → Output: (batch_num, seq_len, vocab_size)
        self.logit = nn.Linear(d_model, self.vocab_size)

    def encode(self, src_embeddings, src_mask):
        # Pass source tokens through num_layers rounds of embedding + encoder
        # Input: (batch_num, input_len) → Output: (batch_num, input_len, model_dim)
        for i in range(self.num_layers):
            # Input: (batch_num, input_len) → Output: (batch_num, input_len, model_dim)
            src_embeddings = self.embed(src_embeddings, i)
            # Input/Output: (batch_num, input_len, model_dim)
            src_embeddings = self.encoder(src_embeddings, src_mask)
        return src_embeddings

    def decode(self, tgt_embeddings, target_mask, src_embeddings, src_mask):
        # Pass target tokens through num_layers rounds of embedding + decoder
        # Input tgt: (batch_num, output_len), src: (batch_num, input_len, model_dim)
        # Output: (batch_num, output_len, model_dim)
        for i in range(self.num_layers):
            # Input: (batch_num, output_len) → Output: (batch_num, output_len, model_dim)
            tgt_embeddings = self.embed(tgt_embeddings, i)
            # Input/Output: (batch_num, output_len, model_dim)
            tgt_embeddings = self.decoder(
                tgt_embeddings, src_embeddings, src_mask, target_mask
            )
        return tgt_embeddings

    def forward(self, src_words, src_mask, target_words, target_mask):
        # Encode source sequence
        # Input: (batch_num, input_len) → Output: (batch_num, input_len, model_dim)
        encoded = self.encode(src_words, src_mask)

        # Decode target sequence conditioned on encoder output
        # Input tgt: (batch_num, output_len), src: (batch_num, input_len, model_dim)
        # Output: (batch_num, output_len, model_dim)
        decoded = self.decode(target_words, target_mask, encoded, src_mask)

        # Project to vocabulary and apply log-softmax
        # Input: (batch_num, output_len, model_dim) → Output: (batch_num, output_len, vocab_size)
        out = F.log_softmax(self.logit(decoded), dim=2)
        return out

# 5) Optimizer

<img src='https://miro.medium.com/max/886/1*ZhGLUwaaqlJ9C0WK0nbAEA.png'>


**Custom learning rate**
$$\Large{lrate = d_{model}^{-0.5} * min(step{\_}num^{-0.5}, step{\_}num * warmup{\_}steps^{-1.5})}$$

[Relationship between Entropy, Cross-Entropy, and KL-Divergence](https://towardsdatascience.com/entropy-cross-entropy-and-kl-divergence-explained-b09cdae917a)

Why **Smoothing** matters:

Without smoothing: The model is trained to be 100% confident, which leads to overconfident predictions, poor calibration, and overfitting.
With smoothing: The model learns to be slightly uncertain, which improves generalization, calibration, and often BLEU scores in translation tasks.

In [ ]:
# Optimizer and label-smoothed loss function definitions

# AdamWarmup: implements the Noam learning rate schedule (warm-up + inverse sqrt decay)
class AdamWarmup:

    def __init__(self, model_size, warmup_steps, optimizer):
        self.model_size = model_size
        self.warmup_steps = warmup_steps
        self.optimizer = optimizer
        self.current_step = 0
        self.lr = 0

    # Compute current learning rate using the Noam formula
    def get_lr(self):
        return (
            self.model_size ** (-0.5) *
            min(self.current_step ** (-0.5), self.current_step * self.warmup_steps ** (-1.5)
            )
        )

    # Advance one step: update lr for all param groups and call optimizer.step()
    def step(self):
        # Increment the number of steps each time we call the step function
        self.current_step += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        # update the learning rate
        self.lr = lr
        self.optimizer.step()

# LossWithLS: KL-divergence loss with label smoothing applied via a soft target distribution
class LossWithLS(nn.Module):

    def __init__(self, size, smooth):
        super(LossWithLS, self).__init__()
        self.criterion = nn.KLDivLoss(size_average=False, reduce=False)
        self.confidence = 1.0 - smooth
        self.smooth = smooth
        self.size = size

    def forward(self, prediction, target, mask):
        # Flatten prediction from (batch_num, output_len, vocab_size) → (batch_num * output_len, vocab_size)
        prediction = prediction.view(-1, prediction.size(-1))

        # Flatten target and mask
        # Input: (batch_num, output_len) → Output: (batch_num * output_len,)
        target = target.contiguous().view(-1)
        mask = mask.float().view(-1)

        # Build smooth label distribution: fill all entries with smooth / (vocab_size - 1)
        # Input/Output: (batch_num * output_len, vocab_size)
        labels = prediction.data.clone()
        labels.fill_(self.smooth / (self.size - 1))

        # Place high-confidence value (1 - smooth) at the ground-truth token index
        # target.unsqueeze(1): (batch_num * output_len, 1)
        labels.scatter_(1, target.data.unsqueeze(1), self.confidence)

        # Compute per-token KL-divergence loss then sum over vocab dim
        # Input: (batch_num * output_len, vocab_size) → Output after sum(1): (batch_num * output_len,)
        loss = self.criterion(prediction, labels)

        # Apply mask and compute mean loss over non-padding tokens
        # Output: scalar
        loss = (loss.sum(1) * mask).sum() / mask.sum()
        return loss

# Instantiate transformer, optimizer with warmup schedule, and label-smoothed loss
transformer = Transformer(d_model=512, heads=8, num_layers=2, word_map=word_map)
transformer = transformer.to('cpu')
adam_optimizer = torch.optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
transformer_optimizer = AdamWarmup(model_size=512, warmup_steps=4000, optimizer=adam_optimizer)
criterion = LossWithLS(len(word_map), 0.2)

# Quick forward pass test to verify output shape
# Input: question (batch_num, max_len), reply_input (batch_num, output_len + 1)
# Output: (batch_num, output_len + 1, vocab_size)
out = transformer(question, question_mask, reply_input, reply_input_mask)
print(out.size())

/usr/local/lib/python3.7/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


torch.Size([32, 26, 18243])


In [ ]:
# Training loop: one epoch of mini-batch gradient descent over the DataLoader
def train(train_loader, transformer, criterion, epoch, device='cpu'):

    transformer.train()
    sum_loss = 0
    count = 0

    for i, (question, reply) in enumerate(train_loader):

        samples = question.shape[0]

        # Move tensors to target device
        # question: (batch_num, input_len),  reply: (batch_num, output_len + 2)
        question = question.to(device)
        reply = reply.to(device)

        # Right-shift reply to produce teacher-forced input and target sequences
        # reply_input:  (batch_num, output_len + 1)
        # reply_target: (batch_num, output_len + 1)
        reply_input = reply[:, :-1]
        reply_target = reply[:, 1:]

        # Build all three masks
        # question_mask:     (batch_num, 1, 1, input_len)
        # reply_input_mask:  (batch_num, 1, output_len + 1, output_len + 1)
        # reply_target_mask: (batch_num, output_len + 1)
        question_mask, reply_input_mask, reply_target_mask = create_masks(question, reply_input, reply_target)

        # Forward pass through Transformer
        # Input: question (batch_num, input_len), reply_input (batch_num, output_len + 1)
        # Output: (batch_num, output_len + 1, vocab_size)
        out = transformer(question, question_mask, reply_input, reply_input_mask)

        # Compute label-smoothed KL-divergence loss
        # Input: out (batch_num, output_len + 1, vocab_size), reply_target (batch_num, output_len + 1)
        # Output: scalar
        loss = criterion(out, reply_target, reply_target_mask)

        # Backpropagation and parameter update with warmup scheduler
        transformer_optimizer.optimizer.zero_grad()
        loss.backward()
        transformer_optimizer.step()

        sum_loss += loss.item() * samples
        count += samples

        if i % 100 == 0:
            print("Epoch [{}][{}/{}]	Loss: {:.3f}".format(epoch, i, len(train_loader), sum_loss/count))

In [ ]:
# Greedy decoding / evaluation: auto-regressively generates a reply one token at a time
def evaluate(transformer, question, question_mask, max_len, word_map, device="cpu"):
    """
    Performs Greedy Decoding with a batch size of 1
    """
    # Build reverse mapping from integer index back to word token
    rev_word_map = {v: k for k, v in word_map.items()}
    # Set transformer to evaluation mode (disables dropout)
    transformer.eval()
    start_token = word_map['<start>']

    # Encode the source question once
    # Input: (1, input_len), mask (1, 1, 1, input_len) → Output: (1, input_len, model_dim)
    encoded = transformer.encode(question, question_mask)

    # Initialise decoder input with the <start> token
    # Output: (1, 1)
    words = torch.LongTensor([[start_token]]).to(device)

    # Auto-regressive decoding loop: generate one token per step
    for step in range(max_len - 1):
        size = words.shape[1]

        # Build causal (look-ahead) mask for the currently generated sequence
        # Output: (1, 1, seq_len, seq_len)
        target_mask = torch.triu(torch.ones(size, size)).transpose(0, 1).type(dtype=torch.uint8)
        target_mask = target_mask.to(device).unsqueeze(0).unsqueeze(0)

        # Decode the sequence so far
        # Input: words (1, seq_len), encoded (1, input_len, model_dim)
        # Output: (1, seq_len, model_dim)
        decoded = transformer.decode(words, target_mask, encoded, question_mask)

        # Project only the last time-step to vocabulary logits
        # decoded[:, -1]: (1, model_dim) → predictions: (1, vocab_size)
        predictions = transformer.logit(decoded[:, -1])

        # Greedy selection: pick the token with highest probability
        # Input: (1, vocab_size) → next_word: scalar index
        _, next_word = torch.max(predictions, dim=1)
        next_word = next_word.item()

        # Stop if the <end> token is generated
        if next_word == word_map['<end>']:
            break

        # Append predicted token to the running sequence
        # Input: (1, seq_len) cat (1, 1) → Output: (1, seq_len + 1)
        words = torch.cat([words, torch.LongTensor([[next_word]]).to(device)], dim=1)

    # Convert token-index tensor to a list, then to a readable string
    if words.dim() == 2:
        words = words.squeeze(0)
        words = words.tolist()

    # Strip <start> token and join remaining tokens into a sentence
    sen_idx = [w for w in words if w not in {word_map['<start>']}]
    sentence = ' '.join([rev_word_map[sen_idx[k]] for k in range(len(sen_idx))])

    return sentence

# Interactive inference loop: prompts user for input and prints model response
device = 'cpu'
while(1):
    question = input("Question: ")
    if question == 'quit':
        break
    max_len = input("Maximum Reply Length: ")

    # Tokenize input string into integer IDs and add batch dimension
    # Input: list of (num_tokens,) → Output: (1, input_len)
    enc_qus = [word_map.get(word, word_map['<unk>']) for word in question.split()]
    question = torch.LongTensor(enc_qus).to(device).unsqueeze(0)

    # Create padding mask for the question
    # Input: (1, input_len) → Output: (1, 1, 1, input_len)
    question_mask = (question!=0).to(device).unsqueeze(1).unsqueeze(1)

    # Run greedy decoding and display the result
    sentence = evaluate(transformer, question, question_mask, int(max_len), word_map)
    print(sentence)

Question: what is your name
Maximum Reply Length: 25
begged funky begged funky begged funky begged funky begged funky begged funky begged funky begged funky begged funky begged funky begged funky porters fiction
Question: quit


# 6) Pretrained Transformer

In [ ]:
# BART-based Dataset and DataLoader: tokenize pairs using BartTokenizer for fine-tuning
from transformers import BartTokenizer, BartModel
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
max_len = 25

# Dataset subclass using HuggingFace tokenizer for encoding question/reply pairs
class Dataset(Dataset):

    def __init__(self, pairs):
        self.pairs = pairs
        self.dataset_size = len(self.pairs)

    def __getitem__(self, i):
        # Tokenize and pad question tokens to max_len using BartTokenizer
        # Input: list of tokens → Output dicts with input_ids/attention_mask: (1, max_len) each
        question = tokenizer(" ".join(self.pairs[i][0]), max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")
        reply = tokenizer(" ".join(self.pairs[i][1]), max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")

        # Return squeezed tensors (remove batch dim from tokenizer output)
        # Output: q_input_ids (max_len,), q_attention_mask (max_len,), r_input_ids (max_len,), r_attention_mask (max_len,)
        return question['input_ids'][0], question['attention_mask'][0], reply['input_ids'][0], reply['attention_mask'][0]

    def __len__(self):
        return self.dataset_size

# Initialize DataLoader; batches each tensor to (batch_num, max_len)
train_loader = DataLoader(Dataset(pairs), batch_size=32, shuffle=True, pin_memory=True)

# Fetch a sample batch and verify shapes
# Output: question (batch_num, max_len), q_mask (batch_num, max_len), reply (batch_num, max_len), t_mask (batch_num, max_len)
question, q_mask, reply, t_mask = next(iter(train_loader))
print("Question: ", question.size())
print("Answer: ", reply.size())

Question:  torch.Size([32, 25])
Answer:  torch.Size([32, 25])


In [ ]:
# BART-backed Transformer classifier: uses BART encoder embeddings fed into nn.Transformer with a linear head
class Transformers(nn.Module):

    def __init__(self, num_classes, hidden_dim=512, nheads=8,
                 num_encoder_layers=6, num_decoder_layers=6):
        super().__init__()

        # BART backbone used as a contextual embedding extractor (encoder_last_hidden_state)
        self.backbone = BartModel.from_pretrained("facebook/bart-base")
        # Standard PyTorch Transformer module
        self.transformer = nn.Transformer(
            hidden_dim, nheads, num_encoder_layers, num_decoder_layers)

        # Final projection from hidden_dim to num_classes (vocab_size here)
        # Input: (batch_num, seq_len, hidden_dim) → Output: (batch_num, seq_len, num_classes)
        self.linear_class = nn.Linear(hidden_dim, num_classes)

    def forward(self, src, tgt, src_mask, tgt_mask):
        # Extract BART encoder hidden states for source and permute for nn.Transformer (seq-first)
        # Input: src (batch_num, input_len), src_mask (batch_num, input_len)
        # Output: (input_len, batch_num, hidden_dim)
        src_embed = self.backbone(src, src_mask)['encoder_last_hidden_state'].permute(1, 0, 2)

        # Extract BART encoder hidden states for target and permute
        # Input: tgt (batch_num, output_len), tgt_mask (batch_num, output_len)
        # Output: (output_len, batch_num, hidden_dim)
        tgt_embed = self.backbone(tgt, tgt_mask)['encoder_last_hidden_state'].permute(1, 0, 2)

        # Pass through nn.Transformer (seq-first convention)
        # Input: src_embed (input_len, batch_num, hidden_dim), tgt_embed (output_len, batch_num, hidden_dim)
        # Output: (output_len, batch_num, hidden_dim)
        out = self.transformer(
            src=src_embed,
            tgt=tgt_embed,
            src_key_padding_mask=src_mask,
            tgt_key_padding_mask=tgt_mask)

        # Project to num_classes, permute back to batch-first, and apply log-softmax
        # Input: (output_len, batch_num, hidden_dim) → permute → (batch_num, output_len, hidden_dim)
        # → linear → (batch_num, output_len, num_classes) → log_softmax → (batch_num, output_len, num_classes)
        result = self.linear_class(out).permute(1, 0, 2)
        result = F.log_softmax(result, dim=2)
        return result

# Instantiate with BART-base hidden dim (768) and vocabulary size as num_classes
model = Transformers(len(word_map), hidden_dim=768)

# Execute a forward pass to verify output shape
# Input: question (batch_num, max_len), q_mask (batch_num, max_len), reply (batch_num, max_len), t_mask (batch_num, max_len)
# Output: (batch_num, max_len, vocab_size)
result = model(question, q_mask, reply, t_mask)
print(result.size())